In [1]:
!pip install -q -U "scikit-learn>=1.8" gseapy

In [1]:
import os, sys, json, shutil, urllib.request
os.chdir("/content")
if not os.path.exists("355-mdd-research"):
    !git clone -q https://github.com/jesss77/355-mdd-research.git
os.chdir("/content/355-mdd-research"); sys.path.insert(0, os.getcwd())

# ---------- Step 1: data ----------
!python src/prepare_data.py
!mkdir -p data/annotation
!wget -q -O data/annotation/GPL570.annot.gz https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL570/annot/GPL570.annot.gz
!wget -q -O data/annotation/GPL10558.annot.gz https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL10nnn/GPL10558/annot/GPL10558.annot.gz
!python src/map_genes.py

import numpy as np, pandas as pd, gseapy, sklearn, joblib
from sklearn.model_selection import GridSearchCV, StratifiedKFold
assert tuple(map(int, sklearn.__version__.split(".")[:2])) >= (1, 8), "upgrade scikit-learn + restart"

# ---------- Steps 2-3: Hallmark + coverage ----------
MSIGDB_VERSION = "2024.1.Hs"; PFX = "ENTREZ_"
gmt_name = f"h.all.v{MSIGDB_VERSION}.entrez.gmt"
os.makedirs("data/genesets", exist_ok=True); gmt_path = f"data/genesets/{gmt_name}"
urllib.request.urlretrieve(f"https://data.broadinstitute.org/gsea-msigdb/msigdb/release/{MSIGDB_VERSION}/{gmt_name}", gmt_path)
hallmark = {}
with open(gmt_path) as f:
    for line in f:
        p = line.strip().split("\t"); hallmark[p[0]] = p[2:]
hallmark_pfx = {k: [PFX + g for g in v] for k, v in hallmark.items()}

shared = set(pd.read_csv("metadata/shared_genes.csv", dtype=str)["entrez_gene_id"])
coverage = pd.DataFrame([{"pathway": k, "genes_in_set": len(v),
                          "genes_present": sum(g in shared for g in v),
                          "coverage_pct": round(100 * sum(g in shared for g in v) / len(v), 1)}
                         for k, v in hallmark.items()]).sort_values("coverage_pct")
os.makedirs("outputs/pathway", exist_ok=True)
coverage.to_csv("outputs/pathway/hallmark_coverage.csv", index=False)
print("pathways:", len(hallmark), "| min genes present:", coverage.genes_present.min(),
      "| median coverage:", coverage.coverage_pct.median())

# ---------- Step 4: ssGSEA ----------
COHORTS = ["GSE98793", "GSE32280", "GSE39653"]
def load_expr(c):
    df = pd.read_csv(f"data/processed/{c}_shared_genes.tsv.gz", sep="\t", dtype={"entrez_gene_id": str})
    df = df.drop(columns="gene_symbol").set_index("entrez_gene_id"); df.index = PFX + df.index
    return df
def run_ssgsea(expr):
    r = gseapy.ssgsea(data=expr, gene_sets=hallmark_pfx, outdir=None, sample_norm_method="rank",
                      correl_norm_type="rank", weight=0.25, min_size=15, max_size=500, threads=4, no_plot=True)
    return r.res2d.pivot(index="Name", columns="Term", values="ES").astype(float).loc[expr.columns]

os.makedirs("data/features", exist_ok=True); features = {}
for c in COHORTS:
    s = run_ssgsea(load_expr(c)); s.index.name = "GSM"
    s.to_csv(f"data/features/{c}_hallmark_ssgsea.csv"); features[c] = s
    print(c, s.shape, "NaN:", int(s.isna().sum().sum()))
e = load_expr("GSE32280")
diff = float((run_ssgsea(e.iloc[:, :1]).iloc[0] - features["GSE32280"].iloc[0]).abs().max())
print("per-patient check (should be 0):", diff)

json.dump({"msigdb_collection": "H (Hallmark)", "msigdb_version": MSIGDB_VERSION, "gene_id": "entrez_gene_id",
           "tool": f"gseapy {gseapy.__version__}", "method": "ssGSEA (Barbie et al. 2009)",
           "score_used": "ES (raw; NES not used because it is rescaled using all samples in the batch)",
           "sample_norm_method": "rank", "correl_norm_type": "rank", "weight": 0.25,
           "min_size": 15, "max_size": 500, "pathways": 50,
           "per_sample_independence_check_max_diff": diff,
           "reference_qn_effect_on_scores": "none (max |diff| = 0 in all cohorts; ssGSEA uses within-sample ranks only)",
           "scaling": "none here; StandardScaler fit on training folds inside the shared pipeline"},
          open("outputs/pathway/ssgsea_settings.json", "w"), indent=2)

# ---------- Step 6: pathway model (Jessy's frozen procedure) ----------
from src.nested_cv import (OUTER_SPLITS, INNER_SPLITS, OUTER_RANDOM_STATE, INNER_RANDOM_STATE,
                           CLASSIFICATION_THRESHOLD, build_parameter_grid)
from src.pipeline_factory import build_model_pipeline
from src.modeling import classification_metrics
def make_search(cv):
    return GridSearchCV(build_model_pipeline(selector="passthrough"), build_parameter_grid(),
                        scoring="roc_auc", cv=cv, refit=True, n_jobs=-1, error_score="raise")

man = pd.read_csv("metadata/cohort_manifest.csv", dtype=str)
man = man[(man.cohort == "GSE98793") & (man.include == "yes")]
feat = features["GSE98793"].loc[man.GSM]
X, y = feat.to_numpy(float), man.label.astype(int).to_numpy()

folds = []
for k, (tr, va) in enumerate(StratifiedKFold(OUTER_SPLITS, shuffle=True, random_state=OUTER_RANDOM_STATE).split(X, y), 1):
    s = make_search(StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=INNER_RANDOM_STATE)).fit(X[tr], y[tr])
    m = classification_metrics(y[va], s.best_estimator_.predict_proba(X[va])[:, 1], threshold=CLASSIFICATION_THRESHOLD)
    folds.append({"fold": k, "best_inner_auroc": s.best_score_, "C": s.best_params_["model__C"],
                  "l1_ratio": s.best_params_["model__l1_ratio"], **m})
folds = pd.DataFrame(folds)
summary = {mt: {"mean": float(folds[mt].mean()), "std": float(folds[mt].std(ddof=1))}
           for mt in ["auroc", "f1", "sensitivity", "specificity"]}
final = make_search(StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=INNER_RANDOM_STATE)).fit(X, y)
coef = pd.Series(final.best_estimator_.named_steps["model"].coef_[0], index=feat.columns)

os.makedirs("outputs/pathway_nested_cv", exist_ok=True); os.makedirs("outputs/pathway_final_model", exist_ok=True)
folds.to_csv("outputs/pathway_nested_cv/fold_metrics.csv", index=False)
joblib.dump(final.best_estimator_, "outputs/pathway_final_model/pathway_model.joblib")
coef.sort_values(key=abs, ascending=False).to_csv("outputs/pathway_final_model/coefficients.csv")
json.dump({"nested_cv": summary,
           "final_model": {"best_inner_auroc": float(final.best_score_), "C": final.best_params_["model__C"],
                           "l1_ratio": final.best_params_["model__l1_ratio"], "nonzero_pathways": int((coef != 0).sum())},
           "sklearn": sklearn.__version__}, open("outputs/pathway_nested_cv/summary.json", "w"), indent=2)
print("\nNested CV AUROC: %.3f ± %.3f (should be 0.585 ± 0.089)" % (summary["auroc"]["mean"], summary["auroc"]["std"]))

# ---------- Package for upload ----------
os.makedirs("results/pathway", exist_ok=True)
for f in ["outputs/pathway/ssgsea_settings.json", "outputs/pathway/hallmark_coverage.csv",
          "outputs/pathway_nested_cv/summary.json", "outputs/pathway_nested_cv/fold_metrics.csv",
          "outputs/pathway_final_model/coefficients.csv"]:
    shutil.copy(f, "results/pathway/")
!zip -qr /content/pathway_push.zip data/features data/genesets results/pathway
from google.colab import files
files.download("/content/pathway_push.zip")

GSE98793: downloading GEO series matrix...
GSE98793: 192 selected: 128 MDD / 64 controls; excluded 0
  54675 probes; selected values [1.131, 14.390]
GSE32280: downloading GEO series matrix...
GSE32280: 16 selected: 8 MDD / 8 controls; excluded 21
  54675 probes; selected values [1.461, 13.789]
  13 metadata warning(s), see combined manifest
GSE39653: downloading GEO series matrix...
GSE39653: 45 selected: 21 MDD / 24 controls; excluded 8
  43116 probes; selected values [6.445, 14.700]

ONE manifest: metadata/cohort_manifest.csv
Audit summary: metadata/dataset_summary.json
GPL10558: 31,253 unambiguous probe mappings, 20,750 genes
GPL570: 42,904 unambiguous probe mappings, 20,848 genes
GSE98793: 192 samples, 42,904/54,675 probes mapped, 20,848 genes
GSE32280: 16 samples, 42,904/54,675 probes mapped, 20,848 genes
GSE39653: 45 samples, 28,446/43,116 probes mapped, 19,457 genes
Shared genes across all three cohorts: 16,467
  wrote data/processed/GSE98793_shared_genes.tsv.gz
  wrote data/pro

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>